# Forming LightRAG database

In [ ]:
# Colab: install pinned, compatible versions
!pip -q install -U pip
!pip -q install "lightrag-hku>=1.2.5" "raganything>=0.1.7" "docling>=2.8.0" "openai>=1.40.0" pillow tqdm python-dotenv
# Optional alternate parser for OCR fallback:
# !pip -q install "mineru>=0.1.7"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'pylatexenc' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'pylatexenc'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [ ]:
from google.colab import userdata
API_KEY=userdata.get('api-key-MicRisk')

In [ ]:
#pdf_path='/content/Schaffner_2020.pdf'
folder_path='/content/documents'

In [ ]:
import asyncio, os, shutil
from raganything import RAGAnything, RAGAnythingConfig
from lightrag.llm.openai import openai_complete_if_cache, openai_embed
from lightrag.utils import EmbeddingFunc

# ---- paths & models ----
WORKING_DIR = "rag_storage"
OUTPUT_DIR  = "output"
TEXT_MODEL   = "gpt-4o"
VISION_MODEL = "gpt-4o"
EMBED_MODEL  = "text-embedding-3-large"

def clear_caches():
    shutil.rmtree(WORKING_DIR, ignore_errors=True)
    shutil.rmtree(OUTPUT_DIR,  ignore_errors=True)
    os.makedirs(WORKING_DIR, exist_ok=True)
    os.makedirs(OUTPUT_DIR,  exist_ok=True)

def make_llm_funcs(api_key: str, base_url: str | None):
    def llm_model_func(prompt, system_prompt=None, history_messages=None, **kwargs):
        return openai_complete_if_cache(
            TEXT_MODEL,
            prompt,
            system_prompt=system_prompt,
            history_messages=history_messages or [],
            api_key=api_key,
            base_url=base_url,
            **kwargs,
        )

    def vision_model_func(prompt, system_prompt=None, history_messages=None, image_data=None, messages=None, **kwargs):
        if messages:
            result = openai_complete_if_cache(
                VISION_MODEL, "", system_prompt=None, history_messages=[],
                messages=messages, api_key=api_key, base_url=base_url, **kwargs
            )
            return result if result is not None else ""  # Return empty string if result is None
        elif image_data:
            mm = []
            if system_prompt:
                mm.append({"role": "system", "content": system_prompt})
            mm.append({
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_data}"}},
                ],
            })
            result = openai_complete_if_cache(
                VISION_MODEL, "", system_prompt=None, history_messages=[],
                messages=mm, api_key=api_key, base_url=base_url, **kwargs
            )
            return result if result is not None else "" # Return empty string if result is None
        else:
            return llm_model_func(prompt, system_prompt, history_messages, **kwargs)


    embedding_func = EmbeddingFunc(
        embedding_dim=3072,
        max_token_size=8192,
        func=lambda texts: openai_embed(
            texts, model=EMBED_MODEL, api_key=api_key, base_url=base_url
        ),
    )
    return llm_model_func, vision_model_func, embedding_func

async def run_docling(pdf_path: str):
    api_key  = API_KEY
    base_url = os.environ.get("OPENAI_BASE_URL")  # optional
    assert api_key, "Missing OPENAI_API_KEY."

    clear_caches()

    config = RAGAnythingConfig(
        working_dir=WORKING_DIR,
        parser="docling",          # robust figure+caption extraction
        parse_method="auto",       # use "ocr" for scanned PDFs
        enable_image_processing=True,
        enable_table_processing=True,
        enable_equation_processing=True,
    )

    llm_fn, vis_fn, emb_fn = make_llm_funcs(api_key, base_url)
    rag = RAGAnything(config=config, llm_model_func=llm_fn, vision_model_func=vis_fn, embedding_func=emb_fn)

    print(">> Parsing PDF with Docling ...")
    try:
        #await rag.process_document_complete(file_path=pdf_path, output_dir=OUTPUT_DIR, parse_method="auto")
        await rag.process_folder_complete(folder_path=pdf_path,output_dir=OUTPUT_DIR, file_extensions=[".pdf", ".docx", ".pptx"], recursive=True, max_workers=4,
                                           parse_method="auto")
    except asyncio.CancelledError:
        print(">> Document processing was cancelled.")
        return # Exit the function if cancelled

    # qs = [
    #     "What are the findings in Figure 4?",
    #     "What are the findings in Fig. 4 (modelled distribution of Ambrosia and Ophraella communa, and expected beetle generations)?",
    # ]
    # for q in qs:
    #     print(f"\n>> Q: {q}")
    #     ans = await rag.aquery(q, mode="hybrid")
    #     print(ans)

await asyncio.sleep(0)  # make sure event loop is ready in Colab
await run_docling(folder_path)

INFO: RAGAnything initialized with config:
INFO:   Working directory: rag_storage
INFO:   Parser: docling
INFO:   Parse method: auto
INFO:   Multimodal processing - Image: True, Table: True, Equation: True
INFO:   Max concurrent files: 1


>> Parsing PDF with Docling ...


Streaming output truncated to the last 5000 lines.
INFO: Merged: `Campylobacter Microbiological Criteria Analysis Table (table)`~`Microbiological Criteria` | 1+1
INFO: Merged: `Campylobacter`~`Campylobacter Microbiological Criteria Analysis Table (table)` | 1+1
INFO: Merged: `Image Path`~`International Journal of Food Microbiology Cover (image)` | 1+1
INFO:  == LLM cache == saving: default:extract:706ac7f31cfc1223d12e18b7cd45f5fe
INFO: Chunk 5 of 12 extracted 12 Ent + 6 Rel chunk-7f0736b2381945b51c262930884b4be1
INFO: Merged: `Cover Page`~`International Journal of Food Microbiology Cover (image)` | 1+1
INFO: Merged: `Broiler Meat`~`Campylobacter Microbiological Criteria Analysis Table (table)` | 1+1
INFO: image processing complete: Campylobacter Control Process Flowchart (image)
INFO: Processing item 7/43: image content
INFO: Merged: `Campylobacter`~`Chicken Meat` | 1+1
INFO:  == LLM cache == saving: default:extract:067b846dcc39c186057af402f9c46422
INFO: Merged: `International Journal 

# Quering database

In [ ]:
import asyncio
import os
from raganything import RAGAnything, RAGAnythingConfig
from lightrag import LightRAG
from lightrag.llm.openai import openai_complete_if_cache, openai_embed
from lightrag.utils import EmbeddingFunc

# ---- paths & models ----
WORKING_DIR  = "rag_storage"      # your existing storage folder
TEXT_MODEL   = "gpt-5"
VISION_MODEL = "gpt-5"
EMBED_MODEL  = "text-embedding-3-large"

def make_llm_funcs(api_key: str, base_url: str | None):
    def llm_model_func(prompt, system_prompt=None, history_messages=None, **kwargs):
        return openai_complete_if_cache(
            TEXT_MODEL,
            prompt,
            system_prompt=system_prompt,
            history_messages=history_messages or [],
            api_key=api_key,
            base_url=base_url,
            **kwargs,
        )

    def vision_model_func(prompt, system_prompt=None, history_messages=None, image_data=None, messages=None, **kwargs):
        if messages:
            result = openai_complete_if_cache(
                VISION_MODEL,
                "",
                system_prompt=None,
                history_messages=[],
                messages=messages,
                api_key=api_key,
                base_url=base_url,
                **kwargs
            )
            return result if result is not None else ""
        elif image_data:
            mm = []
            if system_prompt:
                mm.append({"role": "system", "content": system_prompt})
            mm.append({
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_data}"}}
                ],
            })
            result = openai_complete_if_cache(
                VISION_MODEL,
                "",
                system_prompt=None,
                history_messages=[],
                messages=mm,
                api_key=api_key,
                base_url=base_url,
                **kwargs
            )
            return result if result is not None else ""
        else:
            return llm_model_func(prompt, system_prompt, history_messages, **kwargs)

    embedding_func = EmbeddingFunc(
        embedding_dim=3072,
        max_token_size=8192,
        func=lambda texts: openai_embed(
            texts, model=EMBED_MODEL, api_key=api_key, base_url=base_url
        ),
    )
    return llm_model_func, vision_model_func, embedding_func

async def load_existing_rag(api_key: str, base_url: str | None):
    # Setup LLM / embeddings
    llm_fn, vis_fn, emb_fn = make_llm_funcs(api_key, base_url)

    # Create the underlying LightRAG instance pointing to the existing folder
    lightrag = LightRAG(
        working_dir=WORKING_DIR,
        llm_model_func=llm_fn,
        embedding_func=emb_fn
    )

    # IMPORTANT: initialize storages (and pipeline status if needed) so that existing storage is loaded. :contentReference[oaicite:0]{index=0}
    await lightrag.initialize_storages()
    # Depending on version: you may need to call `await initialize_pipeline_status()` here.
    # from lightrag.kg.shared_storage import initialize_pipeline_status
    # await initialize_pipeline_status()

    # Now wrap it in RAGAnything with vision capability
    rag = RAGAnything(
        lightrag = lightrag,
        vision_model_func = vis_fn
    )

    return rag

# Optionally: helper to query in one function
async def ask_question(rag, question: str, mode: str = "hybrid"):
    return await rag.aquery(question, mode=mode)


In [ ]:
import os
import asyncio

#API key & base URL
api_key  = API_KEY   # or set directly: api_key = "your-key"
base_url = os.environ.get("OPENAI_BASE_URL")

# Load the RAG instance
rag = await load_existing_rag(api_key, base_url)
#rag = await load_existing_rag(api_key)

questions = [
    """Based on the literature in your database you need to provide descriptions for stakeholders involved in the following discussion topic

    Problem definition: How best to balance the trade-offs for setting a food safety microbiological criteria for Campylobacter spp in broiler meat according to the EU legislation No 2073/2005?  ",
    Rationale: highly relevant food-borne pathogen (ranked 1 in the EU), steadily to increasing number of human incidence in the EU (and abroad), food hygiene criterion set for Campylobacter on broiler carcases in 2017, several factors influence prevalence at various stages within the food chain, current intervention strategies fail to combat human case rates

    Expected direction of negotiations:

    Main area:
      •	Will the establishment of a food safety microbiological criteria for Campylobacter spp in broiler meat (replacing the current process hygiene criterion) according to the EU legislation No 2073/2005 improve food safety and public health - or will the additional costs for industry and farmers, and related to food waste outweigh the public health gains?
    Side areas:
      •	What are possible animal welfare implications?
      • How may consumers' expectations and perceptions interfere - balancing price of meat vs. food safety?

    Scientific evidence to be considered during the workshop:


    •	Scientific evidence papers
    •	Industry data (if available)
    •	Publicly available reports
    •	Published data

    Your task is to provide the descriptions of the following stakeholders based on the literature from your database

    •	Environmental expert (food waste)
    •	Food Safety Authority
    •	Industry
    •	Consumer

    You need to provide the description of each stakeholders by filling the following categories:

     Description
     Short-Term Goals
     General Aim
     Scientific Foundation
     Ethical Foundation
     References

    Cite the used literature in APA6 format with both in-text citations and bibliography at the Reference section. Use only the materials from your database.
    Use as much factual information and cite as much useful sources as possible, but do not repeat yourself.

     """

]

for q in questions:
    print(f"\n>> Q: {q}")
    ans = await ask_question(rag, q, mode="hybrid")
    print("Answer:", ans)

INFO: [_] Loaded graph from /content/rag_storage/graph_chunk_entity_relation.graphml with 8815 nodes, 7388 edges
INFO: RAGAnything initialized with config:
INFO:   Working directory: ./rag_storage
INFO:   Parser: mineru
INFO:   Parse method: auto
INFO:   Multimodal processing - Image: True, Table: True, Equation: True
INFO:   Max concurrent files: 1



>> Q: Based on the literature in your database you need to provide descriptions for stakeholders involved in the following discussion topic 

    Problem definition: How best to balance the trade-offs for setting a food safety microbiological criteria for Campylobacter spp in broiler meat according to the EU legislation No 2073/2005?  ",
    Rationale: highly relevant food-borne pathogen (ranked 1 in the EU), steadily to increasing number of human incidence in the EU (and abroad), food hygiene criterion set for Campylobacter on broiler carcases in 2017, several factors influence prevalence at various stages within the food chain, current intervention strategies fail to combat human case rates

    Expected direction of negotiations: 

    Main area: 
      •	Will the establishment of a food safety microbiological criteria for Campylobacter spp in broiler meat (replacing the current process hygiene criterion) according to the EU legislation No 2073/2005 improve food safety and public h

INFO: Parser 'mineru' installation verified
INFO: Initializing parse cache for pre-provided LightRAG instance
INFO: Multimodal processors initialized with context support
INFO: Available processors: ['image', 'table', 'equation', 'generic']
INFO: Context configuration: ContextConfig(context_window=1, context_mode='page', max_context_tokens=2000, include_headers=True, include_captions=True, filter_content_types=['text'])
INFO: Executing VLM enhanced query: Based on the literature in your database you need to provide descriptions for stakeholders involved ...
INFO: LLM func: 4 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 375s)
INFO:  == LLM cache == saving: hybrid:keywords:c23b5e01b211379b4e9a7d66d67db251
INFO: Query nodes: Campylobacter spp., Broiler meat, EU legislation No 2073/2005, Food safety microbiological criteria, Process hygiene criterion, Food hygiene criterion (2017), Broiler carcasses, Industry and farmers, Food waste, Consumer expectations, Pri

Answer: Below are concise, evidence-grounded profiles for four key stakeholder groups to support a negotiation on whether to replace the current process hygiene criterion (PHC) with a food safety microbiological criterion (MC) for Campylobacter spp. in broiler meat under Commission Regulation (EC) No 2073/2005, as amended by Commission Regulation (EU) 2017/1495. The descriptions focus on roles, goals, and value frameworks, anchored in quantitative microbial risk assessment (QMRA), EU regulatory context, and risk-based control literature.

Stakeholder: Environmental expert (food waste)

- Description
  - Focuses on the system-level consequences of changing criteria (from PHC to MC) for Campylobacter control on broiler carcasses, emphasizing the need to translate surveillance data and model outputs into risk-informed, proportionate actions along the chain. Prioritizes approaches that target risk “at source” (farm/slaughterhouse) in a cost-efficient manner to avoid unnecessary downstream 

# Inspecting KG in Neo4J (Optional, not directly used in the manuscript)

In [ ]:
!pip install neo4j # Property graph is recommended to use

import os
import json
import xml.etree.ElementTree as ET
from neo4j import GraphDatabase

In [ ]:
# @title Graph-RAG visualization. GRAPHML - JSON

# Constants
WORKING_DIR = "/content/rag_storage"
BATCH_SIZE_NODES = 500
BATCH_SIZE_EDGES = 100

# Neo4j connection credentials
NEO4J_URI=userdata.get('neo4j-uri-seminar')
NEO4J_USERNAME="neo4j"
NEO4J_PASSWORD=userdata.get('neo4j-password')


def xml_to_json(xml_file):
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()

        # Print the root element's tag and attributes to confirm the file has been correctly loaded
        print(f"Root element: {root.tag}")
        print(f"Root attributes: {root.attrib}")

        data = {"nodes": [], "edges": []}

        # Use namespace
        namespace = {"": "http://graphml.graphdrawing.org/xmlns"}

        for node in root.findall(".//node", namespace):
            node_data = {
                "id": node.get("id").strip('"'),
                "entity_type": node.find("./data[@key='d1']", namespace).text.strip('"')
                if node.find("./data[@key='d1']", namespace) is not None
                else "",
                "description": node.find("./data[@key='d2']", namespace).text
                if node.find("./data[@key='d2']", namespace) is not None
                else "",
                "source_id": node.find("./data[@key='d3']", namespace).text
                if node.find("./data[@key='d3']", namespace) is not None
                else "",
            }
            data["nodes"].append(node_data)

        for edge in root.findall(".//edge", namespace):
            edge_data = {
                "source": edge.get("source").strip('"'),
                "target": edge.get("target").strip('"'),
                "weight": float(edge.find("./data[@key='d5']", namespace).text)
                if edge.find("./data[@key='d5']", namespace) is not None
                else 0.0,
                "description": edge.find("./data[@key='d6']", namespace).text
                if edge.find("./data[@key='d6']", namespace) is not None
                else "",
                "keywords": edge.find("./data[@key='d7']", namespace).text
                if edge.find("./data[@key='d7']", namespace) is not None
                else "",
                "source_id": edge.find("./data[@key='d8']", namespace).text
                if edge.find("./data[@key='d8']", namespace) is not None
                else "",
            }
            data["edges"].append(edge_data)

        # Print the number of nodes and edges found
        print(f"Found {len(data['nodes'])} nodes and {len(data['edges'])} edges")

        return data
    except ET.ParseError as e:
        print(f"Error parsing XML file: {e}")
        return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None


def convert_xml_to_json(xml_path, output_path):
    """Converts XML file to JSON and saves the output."""
    if not os.path.exists(xml_path):
        print(f"Error: File not found - {xml_path}")
        return None

    json_data = xml_to_json(xml_path)
    if json_data:
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(json_data, f, ensure_ascii=False, indent=2)
        print(f"JSON file created: {output_path}")
        return json_data
    else:
        print("Failed to create JSON data")
        return None


def process_in_batches(tx, query, data, batch_size):
    """Process data in batches and execute the given query."""
    for i in range(0, len(data), batch_size):
        batch = data[i : i + batch_size]
        tx.run(query, {"nodes": batch} if "nodes" in query else {"edges": batch})


xml_file  = os.path.join(WORKING_DIR, "graph_chunk_entity_relation.graphml")
json_file = os.path.join(WORKING_DIR, "graph_data.json")

json_data = convert_xml_to_json(xml_file, json_file)
if json_data is None:
    raise RuntimeError("Failed to convert GraphML → JSON")
print(f"Loaded {len(json_data['nodes'])} nodes & {len(json_data['edges'])} edges")

Root element: {http://graphml.graphdrawing.org/xmlns}graphml
Root attributes: {'{http://www.w3.org/2001/XMLSchema-instance}schemaLocation': 'http://graphml.graphdrawing.org/xmlns http://graphml.graphdrawing.org/xmlns/1.0/graphml.xsd'}
Found 8815 nodes and 7388 edges
JSON file created: /content/rag_storage/graph_data.json
Loaded 8815 nodes & 7388 edges


In [ ]:
# @title Uploading data to Neo4J with Cypher queries
create_nodes_query = """
UNWIND $nodes AS node
MERGE (e:Entity {id: node.id})
SET e.entity_type = node.entity_type,
    e.description   = node.description,
    e.source_id     = node.source_id,
    e.displayName   = node.id
REMOVE e:Entity
WITH e, node
CALL apoc.create.addLabels(e, [node.id]) YIELD node AS labeledNode
RETURN count(*)
"""

create_edges_query = """
UNWIND $edges AS edge
MATCH (source {id: edge.source})
MATCH (target {id: edge.target})
WITH source, target, edge,
     CASE
        WHEN edge.keywords CONTAINS 'lead'        THEN 'lead'
        WHEN edge.keywords CONTAINS 'participate' THEN 'participate'
        WHEN edge.keywords CONTAINS 'uses'        THEN 'uses'
        WHEN edge.keywords CONTAINS 'located'     THEN 'located'
        WHEN edge.keywords CONTAINS 'occurs'      THEN 'occurs'
       ELSE REPLACE(SPLIT(edge.keywords, ',')[0], '\"', '')
     END AS relType
CALL apoc.create.relationship(
    source, relType,
    {
      weight:      edge.weight,
      description: edge.description,
      keywords:    edge.keywords,
      source_id:   edge.source_id
    },
    target
) YIELD rel
RETURN count(*)
"""

set_displayname_and_labels_query = """
MATCH (n)
SET n.displayName = n.id
WITH n
CALL apoc.create.setLabels(n, [n.entity_type]) YIELD node
RETURN count(*)
"""

nodes = json_data.get("nodes", [])
edges = json_data.get("edges", [])

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)


try:
    with driver.session() as session:
        # Load nodes
        session.execute_write(
            process_in_batches,
            create_nodes_query,
            nodes,
            BATCH_SIZE_NODES
        )
        print("Nodes imported ✅")

        # Load edges
        session.execute_write(
            process_in_batches,
            create_edges_query,
            edges,
            BATCH_SIZE_EDGES
        )
        print("Edges imported ✅")

        # Final label & displayName fix
        session.run(set_displayname_and_labels_query)
        print("Display names & labels set ✅")

except Exception as e:
    print(f"Error during import: {e}")
finally:
    driver.close()


Nodes imported ✅
Edges imported ✅
Display names & labels set ✅
